In [1]:
import sys
sys.path.insert(0, '../lib')

In [2]:
import pathlib
import joblib
import os

import numpy as np
import pandas as pd
import statsmodels.stats.multitest
import openpyxl

import common_data
import common_plots

In [ ]:
DEG_PREFIXES = ['12', '20', '31', '40', '50']

In [ ]:
BASE = pathlib.Path('../08_website/explore_dea/')

In [ ]:
SUPP_BASE = pathlib.Path(common_data.DATA / '05_pseudobulk/70c_export')

In [6]:
DATA_FILES = {}
for f in os.listdir('.'):
    if f.endswith('.joblib'):
        DATA_FILES[f.split('_')[0][:2]] = f

In [ ]:
PADJ_CUTOFF = 0.05
class ComparisonInfo:
    def __init__(self, control, condition, genes, genes_to_keep):
        self.control = control
        self.condition = condition
        self.genes_raw = genes
        self.filter_genes(genes_to_keep)

    def filter_genes(self, genes_to_keep):
        filtered_degs = self.genes_raw.loc[self.genes_raw.index.isin(genes_to_keep), :].copy()
        filtered_degs = filtered_degs.loc[filtered_degs.pvalue.notna()].copy()
        filtered_degs['padj'] = statsmodels.stats.multitest.fdrcorrection(
            filtered_degs.pvalue,
            alpha=PADJ_CUTOFF
        )[1]
        # recompute gene status based on new `padj`
        filtered_degs['sign'] = ''
        filtered_degs.loc[
            filtered_degs.padj.lt(PADJ_CUTOFF)
            & filtered_degs.log2FoldChange.gt(0),
            'sign'
        ] = f'Up in {self.condition}'
        filtered_degs.loc[
            filtered_degs.padj.lt(PADJ_CUTOFF)
            & filtered_degs.log2FoldChange.lt(0),
            'sign'
        ] = f'Up in {self.control}'
        self.genes = filtered_degs


class CellTypeInfo:
    def __init__(self, path, task_info, genes_to_keep):
        self.path = path
        self.task_info = task_info
        self.comparisons = []
        self.meta = pd.read_csv(path / 'meta.csv', index_col=0)
        self.name = self.meta.cell_type.values[0]

        self.load_comparisons(genes_to_keep)

    def load_comparisons(self, genes_to_keep):
        for run in self.path.glob('**/degs.csv'):
            self.comparisons.append(
                ComparisonInfo(
                    self.task_info.column_values[0],
                    self.task_info.column_values[1],
                    pd.read_csv(run, index_col=0),
                    genes_to_keep[self.name]
                )
            )

    @property
    def n_comparisons(self):
        return len(self.comparisons)

In [8]:
class TaskData:
    def __init__(self, task, task_info):
        self.task = task
        self.task_info = task_info
        self.info = {}

In [9]:
TaskInfo = common_data.TaskInfo

In [10]:
secretome = pd.read_table('protein_class_Predicted.tsv')

In [ ]:
COMPARISON_NAMES = {
    'alive_vs_dead': 'Survived vs. Died (all samples)',
    'alive_vs_dead_bacterial': 'Survived vs. Died (bacterial)',
    'alive_vs_dead_viral': 'Survived vs. Died (early SARS-CoV-2)',
    'alive_vs_dead_pathogen_negative': 'Survived vs. Died (pathogen-negative)',
    'no-vap': 'CAP/HAP/NPC then no VAP',
    'vap': 'CAP/HAP/NPC then VAP',
    'baseline': 'Future No VAP vs. Future VAP',
    'vap_cured': 'VAP will be cured vs. not cured',
    'Healthy vs Mixed': 'Healthy vs. Mixed',
    'Bacteria vs Mixed': 'Bacterial vs. Mixed',
    'Healthy vs Bacteria': 'Healthy vs. Bacterial',
    'NPC vs Bacteria': 'NPC vs. Bacterial',
    'NPC vs Mixed': 'NPC vs. Mixed',
    'Early SARS-CoV-2 vs Bacteria': 'Early SARS-CoV-2 vs. Bacterial',
    'Late SARS-CoV-2 vs Bacteria': 'Late SARS-CoV-2 vs. Bacterial',
    'Early SARS-CoV-2 vs Mixed': 'Early SARS-CoV-2 vs. Mixed',
    'Late SARS-CoV-2 vs Mixed': 'Late SARS-CoV-2 vs. Mixed',
    'Bacteria only vs Mixed': 'Bacterial vs. Mixed',
    'gram+_vs_gram-': 'Gram+ vs. Gram–',
    'pseudomonas_vs_gram-': 'Pseudomonas vs. Other Gram–',
    'age': 'Age < 65 vs. ≥ 65',
    'sex': 'Male vs. Female',
    'immunocompromised': 'Immunocompetent vs. not',
    'steroid_dose': 'Cumulative steroid dose ≤ 600 vs. > 600',
    'days_on_vent': 'Days on ventilator < 2 vs. > 16'
}

In [12]:
ORDER = [
    'Healthy vs. NPC',
    'Healthy vs. Early SARS-CoV-2',
    'Healthy vs. Late SARS-CoV-2',
    'Healthy vs. Bacterial',
    'Healthy vs. Mixed',
    'NPC vs. Early SARS-CoV-2',
    'NPC vs. Late SARS-CoV-2',
    'NPC vs. Bacterial',
    'NPC vs. Mixed',
    'Early SARS-CoV-2 vs. Late SARS-CoV-2',
    'Early SARS-CoV-2 vs. Bacterial',
    'Early SARS-CoV-2 vs. Mixed',
    'Late SARS-CoV-2 vs. Bacterial',
    'Late SARS-CoV-2 vs. Mixed',
    'Bacterial vs. Mixed',
    'Gram+ vs. Gram–',
    'Pseudomonas vs. Other Gram–',

    'Survived vs. Died (all samples)',
    'Survived vs. Died (early SARS-CoV-2)',
    'Survived vs. Died (bacterial)',
    'Survived vs. Died (pathogen-negative)',

    'CAP/HAP/NPC then no VAP',
    'CAP/HAP/NPC then VAP',
    'Future No VAP vs. Future VAP',

    'VAP will be cured vs. not cured',

    'Age < 65 vs. ≥ 65',
    'Male vs. Female',
    'Immunocompetent vs. not',
    'Cumulative steroid dose ≤ 600 vs. > 600',
    'Days on ventilator < 2 vs. > 16'
]

In [25]:
len(ORDER)

30

In [13]:
def task_to_name(slug, info):
    if slug in COMPARISON_NAMES:
        return COMPARISON_NAMES[slug]
    g1 = info.task_info.column_values[0]
    g2 = info.task_info.column_values[1]
    if g1 == 'Bacteria':
        g1 = 'Bacterial'
    if g2 == 'Bacteria':
        g2 = 'Bacterial'
    comp_name = f'{g1} vs. {g2}'
    return comp_name

In [ ]:
def save_degs(data, stats):
    for k, d in data.items():
        comp_name = task_to_name(k, d)
        if comp_name not in ORDER:
            print(f'skipping {comp_name}, not in ORDER')
            print(d.task_info)
            continue
        wb = openpyxl.Workbook()
        del wb['Sheet']
        for ct in d.info.keys():
            ct_path = common_plots.CELL_TYPES_DISPLAY.get(ct.replace('_', ' '), ct.replace('_', ' '))
            if ct_path == 'Perivascular macrophages':
                ct_path = 'Interstitial macrophages'

            stats_idx = stats.task.eq(comp_name) & stats.cell_type.eq(ct_path)
            if stats_idx.sum() != 1:
                print(f'Skipping {ct_path} for {comp_name}, not in stats')
                continue
            deg_path = BASE / comp_name.replace('/', '_') / f'{ct_path}_degs.csv.gz'
            degs = d.info[ct].comparisons[0].genes
            degs['secretome'] = pd.Series(degs.index.isin(secretome.Gene)).replace(
                {True: 'Yes', False: ''}
            ).values
            degs = degs.drop(columns=['lfcSE', 'stat', 'pvalue'])
            degs['baseMean'] = degs['baseMean'].round(5)
            degs['log2FoldChange'] = degs['log2FoldChange'].round(5)
            degs['padj'] = degs['padj'].apply('{:.3e}'.format)
            degs = degs.sort_values('log2FoldChange', ascending=False)

            ws = wb.create_sheet(title=ct_path)
            ws.append(['gene'] + list(degs.columns))
            for i, row in degs.iterrows():
                ws.append([i] + list(row))
            deg_path.parent.mkdir(parents=True, exist_ok=True)
            degs.to_csv(deg_path)
        wb.save(SUPP_BASE / f'Supplementary Table X {comp_name.replace("/", "_")}.xlsx')

In [15]:
def save_gsea(data, gsea_base, stats):
    for k, task in data.items():
        comp_name = task_to_name(k, task)
        if comp_name not in ORDER:
            print(f'skipping {comp_name}, not in ORDER')
            print(task.task_info)
            continue

        gseas = []
        for ct, _ in task.info.items():
            if hasattr(task, 'task_info'):
                task_path = task.task_info.pathname
            else:
                task_path = task.task
            ct_path = common_plots.CELL_TYPES_DISPLAY.get(ct.replace('_', ' '), ct.replace('_', ' '))
            if ct_path == 'Perivascular macrophages':
                ct_path = 'Interstitial macrophages'
            stats_idx = stats.task.eq(comp_name) & stats.cell_type.eq(ct_path)
            if stats_idx.sum() != 1:
                print(f'Skipping {ct} for {task_path} (not in stats)')
                continue
            gsea_path = gsea_base  / task_path / ct / 'gsea.csv'
            if not gsea_path.exists():
                print(f'No GSEA for {ct} in {task_path}')
                continue
            gsea = pd.read_csv(gsea_path, index_col=0)
            gsea['cell_type'] = ct_path
            gseas.append(gsea)
        gsea = pd.concat(gseas)

        gsea_path = BASE / f'{comp_name.replace("/", "_")}_gsea.csv.gz'
        gsea['p-value'] = gsea['NOM p-value'].copy()
        idx = gsea['p-value'].eq(0)
        min_p_val = gsea['p-value'][~idx].min()
        gsea.loc[idx, 'p-value'] = min_p_val * 0.1
        gsea['padj'] = statsmodels.stats.multitest.fdrcorrection(gsea['p-value'])[1]
        gsea['-log10(padj)'] = -np.log10(gsea.padj)
        gsea['-log10(padj)'] *= np.sign(gsea.NES)
        gsea = gsea.drop(columns=['ES', 'NOM p-value', 'FDR p-value', 'Set size', 'Tag %', 'Rank %', 'p-value'])
        gsea.to_csv(gsea_path)

In [ ]:
GSEA_PREFIXES = {
    '12': '12_degs',
    '20': '20c_degs',
    '31': '30b_degs',
    '40': '40c_degs',
    '50': '50c_degs'
}

In [ ]:
stats = pd.read_csv('../08_website/explore_dea/merged_stats.csv.gz', index_col=0)

In [24]:
for prefix in DEG_PREFIXES:
    data = joblib.load(DATA_FILES[prefix])
    save_degs(data, stats)

skipping first vs. second, not in ORDER
TaskInfo(pathname='vap', column='pathogens_coarse', column_values=['first', 'second'], split_column='blah')
skipping first vs. second, not in ORDER
TaskInfo(pathname='no_vap', column='pathogens_coarse', column_values=['first', 'second'], split_column='blah')
skipping no_vap vs. vap, not in ORDER
TaskInfo(pathname='combined', column='pathogens_coarse', column_values=['no_vap', 'vap'], split_column='blah')
Skipping B cells for CAP/HAP/NPC then VAP, not in stats
Skipping Classical monocytes-1 CCR2 for CAP/HAP/NPC then VAP, not in stats
Skipping Plasma cells for CAP/HAP/NPC then VAP, not in stats
Skipping Proliferating CD4 T cells for CAP/HAP/NPC then VAP, not in stats
Skipping Proliferating plasma cells for CAP/HAP/NPC then VAP, not in stats
Skipping B cells for CAP/HAP/NPC then no VAP, not in stats
Skipping Proliferating NUPR1+ AM for CAP/HAP/NPC then no VAP, not in stats
Skipping B cells for Future No VAP vs. Future VAP, not in stats
Skipping Clas

In [18]:
PSEUDOBULK_BASE = pathlib.Path('../data/05_pseudobulk/')
for prefix in DEG_PREFIXES:
    data = joblib.load(DATA_FILES[prefix])
    save_gsea(data, PSEUDOBULK_BASE / GSEA_PREFIXES[prefix], stats)

skipping first vs. second, not in ORDER
TaskInfo(pathname='vap', column='pathogens_coarse', column_values=['first', 'second'], split_column='blah')
skipping first vs. second, not in ORDER
TaskInfo(pathname='no_vap', column='pathogens_coarse', column_values=['first', 'second'], split_column='blah')
skipping no_vap vs. vap, not in ORDER
TaskInfo(pathname='combined', column='pathogens_coarse', column_values=['no_vap', 'vap'], split_column='blah')
Skipping B_cells for combined-vap (not in stats)
Skipping Classical_monocytes-1_CCR2 for combined-vap (not in stats)
Skipping Plasma_cells for combined-vap (not in stats)
Skipping Proliferating_CD4_T_cells for combined-vap (not in stats)
Skipping Proliferating_plasma_cells for combined-vap (not in stats)
Skipping B_cells for combined-no-vap (not in stats)
Skipping Proliferating_NUPR1+_Macs for combined-no-vap (not in stats)
Skipping B_cells for combined-baseline (not in stats)
Skipping Classical_monocytes-1_CCR2 for combined-baseline (not in stat